# Guided tensor lessons

These four lessons use Rainbow Tensor 1.5.0 to connect selected outputs with their sources and local arithmetic. Each code cell is the same source used by the documented lesson checker. Run one section at a time in a notebook with widget support.


## Keep binary operands in order

Predict the sources of output (1, 2) when shapes (2, 1) and (1, 3) broadcast. Check addition, subtraction, multiplication, and division.


In [ ]:
"""Match broadcast source coordinates while keeping binary operand order."""

import numpy as np
from IPython.display import display

import rainbow_tensor as rt

a = np.array([[10], [20]])
b = np.array([[1, 2, 3]])
expected = a + b
addition = rt.explore(rt.add, a, b, focus=(1, 2))
expression = addition.visual.trace.expression
assert addition.result_shape == expected.shape == (2, 3)
assert expression.operator == "add"
assert [(ref.operand, ref.coordinate) for ref in expression.operands] == [
    (0, (1, 0)), (1, (0, 2)),
]
assert addition.visual.trace.terms == ()
assert expected[1, 2] == a[1, 0] + b[0, 2] == 23
display(addition.visual)

addition.set_focus((0, 2))
assert addition.visual.trace.expression.operands[0].coordinate == (0, 0)
assert expected[0, 2] == 13
display(addition)

# A literal is a scalar value. A tuple instead describes generated tensor data.
halved = rt.divide(a, 2, focus=(1, 0))
assert halved.trace.expression.operator == "divide"
assert halved.trace.expression.operands[1].coordinate == ()
display(halved)

flow = rt.Flow()
left = flow.input(a, name="A")
right = flow.input(b, name="B")
two = flow.input(2, name="Two")
assert flow.add(left, right).value((1, 2)) == (a + b)[1, 2] == 23
assert flow.subtract(left, right).value((1, 2)) == (a - b)[1, 2] == 17
assert flow.multiply(left, right).value((1, 2)) == (a * b)[1, 2] == 60
assert flow.divide(left, two).value((1, 0)) == (a / 2)[1, 0] == 10



## Build one contribution at a time

Predict the next subtotal of the mean over [4, 5, 6], then compare the products in a small matrix multiplication.


In [ ]:
"""Separate a mean's subtotal from its divisor, then inspect a dot product."""

import numpy as np
from IPython.display import display

import rainbow_tensor as rt

x = np.arange(1, 7).reshape(2, 3)
mean_lesson = rt.walkthrough(rt.mean, x, axis=1, focus=(1,))
assert mean_lesson.snapshot.numeric_complete
assert mean_lesson.snapshot.factor_values == (4,)
assert mean_lesson.snapshot.subtotal == 4
assert mean_lesson.snapshot.output_value == x.mean(axis=1)[1] == 5
display(mean_lesson.visual)

mean_lesson.next_term()
assert mean_lesson.snapshot.term == 1
assert mean_lesson.snapshot.factor_values == (5,)
assert mean_lesson.snapshot.subtotal == 9
mean_lesson.select_term(2)
assert mean_lesson.snapshot.subtotal == 15
assert mean_lesson.snapshot.step.divisor == 3
assert mean_lesson.snapshot.output_value == 15 / 3
display(mean_lesson.visual)

# A new output starts again at its first contribution.
mean_lesson.set_focus((0,))
assert mean_lesson.snapshot.term == 0
assert mean_lesson.snapshot.factor_values == (1,)
assert mean_lesson.snapshot.output_value == 2
display(mean_lesson)

a = np.array([[1, 2]])
b = np.array([[3], [4]])
product_lesson = rt.walkthrough(rt.matmul, a, b, focus=(0, 0))
assert product_lesson.snapshot.factor_values == (1, 3)
assert product_lesson.snapshot.term_value == 3
product_lesson.next_term()
assert product_lesson.snapshot.factor_values == (2, 4)
assert product_lesson.snapshot.term_value == 8
assert product_lesson.snapshot.subtotal == (a @ b)[0, 0] == 11
display(product_lesson)



## Inspect a row total inside division

Explain 9 / (3 + 6 + 9). The value 9 occurs in two paths. Follow the denominator and explain why that count is not a coefficient.


In [ ]:
"""Follow a numerator and its row total through an explicitly recorded division."""

import numpy as np
from IPython.display import display

import rainbow_tensor as rt

scores = np.array([[2., 4., 6.], [3., 6., 9.]])
expected = scores / scores.sum(axis=1, keepdims=True)

flow = rt.Flow()
source = flow.input(scores, name="Scores")
totals = flow.sum(source, axis=1, keepdims=True, name="Totals")
normalized = flow.divide(source, totals, name="Normalized")
assert totals.shape == (2, 1)
assert normalized.shape == expected.shape == (2, 3)
assert normalized.value((1, 2)) == expected[1, 2] == 9 / 18 == 0.5
np.testing.assert_allclose(expected.sum(axis=1), [1., 1.])

trace = normalized.trace((1, 2))
assert trace.complete
assert trace.steps[0].binary_operator == "divide"
assert {root.reference.coordinate: root.count for root in trace.roots} == {
    (1, 0): 1, (1, 1): 1, (1, 2): 2,
}
normalization_lesson = rt.walkthrough(normalized, focus=(1, 2))
assert normalization_lesson.snapshot.factor_values == (9., 18.)
assert normalization_lesson.snapshot.binary_operator == "divide"
assert normalization_lesson.snapshot.term_value == 0.5
display(normalization_lesson.visual)

# Inspect the denominator's own sum without flattening the surrounding division.
denominator = next(
    index for index, step in enumerate(normalization_lesson.snapshot.trace.steps)
    if step.operation == "sum"
)
normalization_lesson.select_occurrence(denominator)
normalization_lesson.select_term(2)
assert normalization_lesson.snapshot.subtotal == 3 + 6 + 9 == 18
assert normalization_lesson.snapshot.output_value == 18
display(normalization_lesson.visual)

normalization_lesson.set_focus((0, 0))
assert normalization_lesson.snapshot.occurrence == 0
assert normalization_lesson.snapshot.binary_operator == "divide"
assert normalized.value((0, 0)) == 2 / 12
display(normalization_lesson)



## Predict reduction shapes

For shape (2, 3, 4), compare reducing (0, 2) with and without keepdims. Then try no axes and all axes.


In [ ]:
"""Predict how a tuple of reduction axes and keepdims change output coordinates."""

import numpy as np
from IPython.display import display

import rainbow_tensor as rt

x = np.arange(24).reshape(2, 3, 4)
playground = rt.reduction_playground(rt.sum, x, axis=(0, 2), focus=(1,))
assert not playground.revealed
playground.prediction.value = "(3,)"
playground.reveal()
assert playground.visual.result_shape == x.sum(axis=(0, 2)).shape == (3,)
assert playground.visual.trace.output_coord == (1,)
assert playground.visual.trace.term_count == 8
assert x.sum(axis=(0, 2))[1] == 4 + 5 + 6 + 7 + 16 + 17 + 18 + 19 == 92
display(playground.visual)

playground.set_parameters(keepdims=True)
assert not playground.revealed
playground.prediction.value = "(1, 3, 1)"
playground.reveal()
playground.explorer.set_focus((0, 1, 0))
assert playground.visual.result_shape == (1, 3, 1)
assert playground.visual.trace.output_coord == (0, 1, 0)
assert playground.visual.trace.term_count == 8
display(playground.visual)

# An empty axis tuple reduces nothing, preserving every source coordinate.
playground.set_parameters(axis=(), keepdims=False)
assert playground.visual.result_shape == x.shape
assert playground.explorer.focus == (0, 1, 0)
assert playground.visual.trace.term_count == 1
playground.reveal()
display(playground.visual)

# None reduces all axes. Try predicting this scalar shape before revealing it.
playground.set_parameters(axis=None)
assert not playground.revealed
playground.prediction.value = "()"
playground.reveal()
assert playground.visual.result_shape == ()
assert playground.visual.trace.output_coord == ()
assert playground.visual.trace.term_count == 24
assert x.sum() == 276
display(playground)



## When you have finished

Close the controls with `addition.close()`, `mean_lesson.close()`, `product_lesson.close()`, `normalization_lesson.close()`, `playground.close()`. Their current .visual objects can still be saved as SVG figures with a separate .text explanation. A host without widget support can display those static views directly.
